# Pix2Pix — paired image-to-image translation with a conditional GAN

> Tutorial pair for [`pix2pix.py`](pix2pix.py).

## 1. Intuition
Given **aligned pairs** (input image, target image), learn the mapping. A pure
L1/L2 regression produces blurry outputs (it averages over plausible targets); a
pure GAN produces sharp but unfaithful ones. Pix2Pix combines them: a
**conditional GAN** for realism plus an **L1** term for fidelity to the specific
target. The discriminator is a **PatchGAN** that judges local patches, which
sharpens textures.

## 2. Concept (the slide)
- **Generator** $G$: a small **U-Net** (encoder-decoder with skip connections)
  mapping input $a$ to output $G(a)$; skips carry low-level structure across.
- **Discriminator** $D$ is **conditional**: it sees the pair $(a, b)$ and judges
  realism. As a **PatchGAN** it outputs a grid of logits, one per local
  receptive field, so it enforces realism at the patch level.
- **Loss:** conditional adversarial + $\lambda$-weighted L1 reconstruction.
- Toy task here: map a filled rectangle to its photometric inverse on 1x16x16.

## 3. Math derivation — conditional GAN objective + L1

**Conditional adversarial loss.** Both nets are conditioned on the input $a$:
$$\mathcal L_{\text{cGAN}}(G,D)=\mathbb E_{a,b}\big[\log D(a,b)\big]
 +\mathbb E_{a}\big[\log\big(1-D(a,G(a))\big)\big].$$
$D$ maximizes this (learn real pairs vs generated pairs); $G$ minimizes it.
Conditioning on $a$ is the key difference from vanilla GAN: $D$ judges not "is
this a real image?" but "is this a real *translation of $a$*?".

**L1 reconstruction.** Add a term tying $G(a)$ to the true target:
$$\mathcal L_{\text{L1}}(G)=\mathbb E_{a,b}\big[\lVert b-G(a)\rVert_1\big].$$
L1 over L2 because it penalizes errors linearly, encouraging **sharper** outputs
(L2 over-smooths by averaging). It captures low-frequency structure; the
adversarial term supplies high-frequency detail.

**Full objective.**
$$G^\star=\arg\min_{G}\max_{D}\;\mathcal L_{\text{cGAN}}(G,D)
 +\lambda\,\mathcal L_{\text{L1}}(G),\qquad \lambda\approx 100.$$

**PatchGAN.** Rather than one global decision, $D$ outputs an $N\times N$ grid of
logits, each classifying a patch; the loss averages over the grid. This models
the image as a Markov random field of patch-realism, gives sharper textures, and
uses far fewer parameters than a full-image discriminator.

**Why it differs from vanilla GAN.** Vanilla GAN maps noise$\to$image
unconditionally. Pix2Pix maps input$\to$output deterministically (noise is mostly
dropped), and the L1 term plus conditional PatchGAN give faithful, sharp pairs.

## 4. Generator / key component

In [ ]:
# ===== actual implementation from pix2pix.py =====
from __future__ import annotations

import numpy as np

import torch

import torch.nn as nn

SEED = 0

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def make_pairs(n: int = 256, size: int = 16, seed: int = SEED):
    rng = np.random.default_rng(seed)
    inp = np.zeros((n, 1, size, size), dtype=np.float32)
    for i in range(n):
        h = rng.integers(4, 8); w = rng.integers(4, 8)
        ys = rng.integers(0, size - h); xs = rng.integers(0, size - w)
        inp[i, 0, ys:ys + h, xs:xs + w] = 1.0
    tgt = 1.0 - inp                      # the deterministic target mapping
    inp = inp * 2 - 1; tgt = tgt * 2 - 1  # -> [-1, 1]
    return inp.astype(np.float32), tgt.astype(np.float32)

class UNetGenerator(nn.Module):
    def __init__(self, ch: int = 1, ngf: int = 32):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(ch, ngf, 4, 2, 1), nn.LeakyReLU(0.2, True))      # 16->8
        self.enc2 = nn.Sequential(nn.Conv2d(ngf, ngf * 2, 4, 2, 1),
                                  nn.BatchNorm2d(ngf * 2), nn.LeakyReLU(0.2, True))           # 8->4
        self.dec1 = nn.Sequential(nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1),
                                  nn.BatchNorm2d(ngf), nn.ReLU(True))                          # 4->8
        # decoder takes the skip-concatenated feature map (ngf + ngf)
        self.dec2 = nn.Sequential(nn.ConvTranspose2d(ngf * 2, ch, 4, 2, 1), nn.Tanh())        # 8->16

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(x)            # ngf  x 8 x 8
        e2 = self.enc2(e1)           # 2ngf x 4 x 4
        d1 = self.dec1(e2)           # ngf  x 8 x 8
        d1 = torch.cat([d1, e1], dim=1)  # skip connection -> 2ngf x 8 x 8
        return self.dec2(d1)

## 5. Trainer / losses

In [ ]:
# ===== actual implementation from pix2pix.py =====
class PatchDiscriminator(nn.Module):
    def __init__(self, ch: int = 1, ndf: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch * 2, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),          # 16->8
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1),
            nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),                  # 8->4
            nn.Conv2d(ndf * 2, 1, 3, 1, 1),                                   # 4x4 grid of patch logits
        )

    def forward(self, inp: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([inp, out], dim=1))

class Pix2PixTorch:
    def __init__(self, ch: int = 1, lr: float = 2e-4, lambda_l1: float = 100.0):
        torch.manual_seed(SEED)
        self.dev = get_device()
        self.lambda_l1 = lambda_l1
        self.G = UNetGenerator(ch).to(self.dev)
        self.D = PatchDiscriminator(ch).to(self.dev)
        self.optG = torch.optim.Adam(self.G.parameters(), lr=lr, betas=(0.5, 0.999))
        self.optD = torch.optim.Adam(self.D.parameters(), lr=lr, betas=(0.5, 0.999))
        self.bce = nn.BCEWithLogitsLoss()
        self.l1 = nn.L1Loss()

    def fit(self, inp: np.ndarray, tgt: np.ndarray, steps: int = 400, batch: int = 32):
        inp = torch.as_tensor(inp, dtype=torch.float32, device=self.dev)
        tgt = torch.as_tensor(tgt, dtype=torch.float32, device=self.dev)
        self.d_hist, self.g_hist, self.l1_hist = [], [], []
        for _ in range(steps):
            idx = torch.randint(0, len(inp), (batch,), device=self.dev)
            a, b = inp[idx], tgt[idx]      # paired input/target

            # --- D step: (a, b) real -> 1, (a, G(a)) fake -> 0 ---
            fake = self.G(a)
            d_real = self.D(a, b)
            d_fake = self.D(a, fake.detach())
            lossD = 0.5 * (self.bce(d_real, torch.ones_like(d_real))
                           + self.bce(d_fake, torch.zeros_like(d_fake)))
            self.optD.zero_grad(); lossD.backward(); self.optD.step()

            # --- G step: fool D on the pair + match target with L1 ---
            d_gen = self.D(a, fake)
            l1 = self.l1(fake, b)
            lossG = self.bce(d_gen, torch.ones_like(d_gen)) + self.lambda_l1 * l1
            self.optG.zero_grad(); lossG.backward(); self.optG.step()

            self.d_hist.append(lossD.item()); self.g_hist.append(lossG.item())
            self.l1_hist.append(l1.item())
        return self

    @torch.no_grad()
    def generate(self, inp: np.ndarray) -> np.ndarray:
        self.G.eval()
        a = torch.as_tensor(inp, dtype=torch.float32, device=self.dev)
        out = self.G(a).cpu().numpy()
        self.G.train()
        return out

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    inp, tgt = make_pairs(256)
    print(f"data: input {inp.shape}, target {tgt.shape}, range [{inp.min():.1f}, {inp.max():.1f}]")

    gan = Pix2PixTorch().fit(inp, tgt, steps=400, batch=32)
    print(f"D loss trend:  {np.mean(gan.d_hist[:50]):.3f} -> {np.mean(gan.d_hist[-50:]):.3f}")
    print(f"L1 loss trend: {np.mean(gan.l1_hist[:50]):.3f} -> {np.mean(gan.l1_hist[-50:]):.3f}"
          f"  (falling => generator matches the paired target)")

    # Quality proxy: per-pixel L1 between G(input) and the true target.
    ti, tt = make_pairs(64, seed=123)
    pred = gan.generate(ti)
    mae = np.abs(pred - tt).mean()
    print(f"held-out per-pixel MAE = {mae:.3f} (0 = perfect; ~2.0 = worst on [-1,1])")

## 6. Train

In [ ]:
demo()

## 7. Visualization

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import pix2pix as M

inp, tgt = M.make_pairs(256)
gan = M.Pix2PixTorch().fit(inp, tgt, steps=250, batch=32)  # lighter retrain just for the picture
ti, tt = M.make_pairs(8, seed=123)
pred = gan.generate(ti)

fig, axes = plt.subplots(3, 8, figsize=(12, 4.6))
rows = [(ti, "input"), (pred, "G(input)"), (tt, "target")]
for r, (imgs, name) in enumerate(rows):
    for j in range(8):
        axes[r, j].imshow(imgs[j, 0], cmap="gray", vmin=-1, vmax=1)
        axes[r, j].axis("off")
    axes[r, 0].set_title(name, loc="left")
fig.suptitle("Pix2Pix: input -> generated -> ground-truth target")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Pix2Pix = conditional GAN (realism) + L1 (fidelity). L1 alone blurs; GAN alone
  drifts; together they give sharp, correct translations.
- PatchGAN judges local patches for sharper textures with fewer parameters;
  the U-Net's skip connections preserve spatial structure.
- Pitfalls: needs **paired** data (use CycleGAN otherwise); $\lambda$ too small
  drifts from the target, too large reverts to blurry regression; output noise is
  usually injected via dropout, not a noise vector.